# 03 - Preprocessing

This notebook prepares the raw sensor dataset for model training by removing null activity rows, inspecting missing values, and scaling features.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import os

RAW_PATH = Path("..") / "data" / "raw" / "mhealth_raw_data.csv"
OUTPUT_PATH = Path("..") / "data" / "interim"

sensor_columns = [
    "alx", "aly", "alz",
    "glx", "gly", "glz",
    "arx", "ary", "arz",
    "grx", "gry", "grz",
]

ACTIVITY_MAP = {
    0: "Null (no activity)",
    1: "Standing still",
    2: "Sitting and relaxing",
    3: "Lying down",
    4: "Walking",
    5: "Climbing stairs",
    6: "Waist bends forward",
    7: "Frontal elevation of arms",
    8: "Knees bending (crouching)",
    9: "Cycling",
    10: "Jogging",
    11: "Running",
    12: "Jump front & back",
}

df = pd.read_csv(RAW_PATH)
df["ActivityName"] = df["Activity"].map(ACTIVITY_MAP)

## Inspect Raw Data

In [ ]:
print("Raw shape:", df.shape)
print("Unique activities:", sorted(df["Activity"].unique()))
print("Missing values per column:\n", df.isna().sum())

df.head()

## Remove Null Activity Rows

In [ ]:
df_clean = df[df["Activity"] != 0].reset_index(drop=True)
print("Shape after removing Activity 0:", df_clean.shape)
print("Remaining activity labels:", sorted(df_clean["Activity"].unique()))

df_clean["ActivityName"] = df_clean["Activity"].map(ACTIVITY_MAP)
df_clean.head()

## Feature Scaling

In [ ]:
scaler = StandardScaler()
df_clean[sensor_columns] = scaler.fit_transform(df_clean[sensor_columns])

scaled_preview = df_clean[sensor_columns].describe().loc[["mean", "std"]]
print(scaled_preview)

## Save Preprocessed Dataset

In [ ]:
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(OUTPUT_PATH / "mhealth_cleaned.csv", index=False)
print(f"Saved cleaned data to {OUTPUT_PATH / 'mhealth_cleaned.csv'}")